In [ ]:
%pip install torch datasets numpy matplotlib

In [ ]:

# Import required libraries

import torch
import torch.nn as nn
from torch.nn import functional as F

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

import random
import os
from pathlib import Path

print("All libraries imported successfully!")


All libraries imported successfully!


In [ ]:

# Set random seeds for reproducibility

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Select available computing device

if torch.backends.mps.is_available():
    device = 'mps'

elif torch.cuda.is_available():
    device = 'cuda'

else:
    device = 'cpu'

print("Using device:", device)


Using device: cuda


In [ ]:

# Load training dataset

train_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train[:20000]"
)

# Load validation dataset

validation_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="validation[:2000]"
)

# Extract stories

train_stories = [
    story["text"]
    for story in train_dataset
    if story["text"].strip()
]

validation_stories = [
    story["text"]
    for story in validation_dataset
    if story["text"].strip()
]

# Display dataset information

print("Training stories:", len(train_stories))
print("Validation stories:", len(validation_stories))

print("\nSample Story:\n")
print(train_stories[0])


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Training stories: 20000
Validation stories: 2000

Sample Story:

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [ ]:
# Create vocabulary from training and validation data

train_text = "".join(train_stories)
validation_text = "".join(validation_stories)

chars = sorted(list(set(train_text + validation_text)))

# Total number of unique characters
vocab_size = len(chars)

# Character to integer mapping
stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to character mapping
itos = {i: ch for i, ch in enumerate(chars)}

# Encoder: Convert text into numbers
def encode(text):
    return [stoi[ch] for ch in text]

# Decoder: Convert numbers back into text
def decode(tokens):
    return ''.join([itos[i] for i in tokens])


# Display vocabulary information

print("Vocabulary size:", vocab_size)

print("\nVocabulary characters:")
print(chars)

# Test tokenizer

sample_text = "Hello world!"

encoded_text = encode(sample_text)
decoded_text = decode(encoded_text)

print("\nOriginal text:", sample_text)
print("Encoded text:", encoded_text)
print("Decoded text:", decoded_text)

Vocabulary size: 98

Vocabulary characters:
['\n', ' ', '!', '"', '$', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\xa0', '¡', '¦', '©', '«', '±', '³', '»', 'Â', 'Ã', 'â', 'œ', 'Š', '˜', '“', '”', '€', '™']

Original text: Hello world!
Encoded text: [35, 58, 65, 65, 68, 1, 76, 68, 71, 65, 57, 2]
Decoded text: Hello world!


In [ ]:

# Encode training and validation text

train_ids = encode(train_text)
validation_ids = encode(validation_text)

# Convert encoded data into PyTorch tensors

train_data = torch.tensor(
    train_ids,
    dtype=torch.long
)

validation_data = torch.tensor(
    validation_ids,
    dtype=torch.long
)

# Display tensor information

print("Training tensor shape:", train_data.shape)
print("Validation tensor shape:", validation_data.shape)

print("\nFirst 20 training tokens:")
print(train_data[:20])


Training tensor shape: torch.Size([17890906])
Validation tensor shape: torch.Size([1591942])

First 20 training tokens:
tensor([42, 67, 58,  1, 57, 54, 78, 11,  1, 54,  1, 65, 62, 73, 73, 65, 58,  1,
        60, 62])


In [ ]:

# GPT Model Configuration

batch_size = 32
block_size = 128

max_iters = 3000
eval_interval = 300
learning_rate = 3e-4

eval_iters = 50

n_embd = 128
n_head = 4
n_layer = 4

dropout = 0.1

print("GPT Configuration")
print("------------------------")
print("Vocabulary size:", vocab_size)
print("Batch size:", batch_size)
print("Block size:", block_size)
print("Embedding dimension:", n_embd)
print("Attention heads:", n_head)
print("Transformer layers:", n_layer)
print("Dropout:", dropout)
print("Training iterations:", max_iters)


GPT Configuration
------------------------
Vocabulary size: 98
Batch size: 32
Block size: 128
Embedding dimension: 128
Attention heads: 4
Transformer layers: 4
Dropout: 0.1
Training iterations: 3000


In [ ]:

def get_batch(split):

    # Select training or validation data
    data = train_data if split == "train" else validation_data

    # Generate random starting positions
    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    # Create input sequences
    x = torch.stack([
        data[i:i + block_size] for i in ix
    ])

    # Create target sequences (shifted by one character)
    y = torch.stack([
        data[i + 1:i + block_size + 1] for i in ix
    ])

    # Move tensors to selected device
    x = x.to(device)
    y = y.to(device)

    return x, y


# Test the batch function
xb, yb = get_batch("train")

print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)

print("\nExample input:")
print(decode(xb[0].tolist()))

print("\nExample target:")
print(decode(yb[0].tolist()))


Input batch shape: torch.Size([32, 128])
Target batch shape: torch.Size([32, 128])

Example input:
irling around, feeling her muscle with each movement. She clapped her hands and smiled. She had never felt so happy and gentle b

Example target:
rling around, feeling her muscle with each movement. She clapped her hands and smiled. She had never felt so happy and gentle be


In [ ]:

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        # Key, Query and Value layers
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Causal mask: prevents looking at future characters
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        # Generate keys and queries
        k = self.key(x)
        q = self.query(x)

        # Calculate attention scores
        weights = q @ k.transpose(-2, -1)

        # Scale attention scores
        weights = weights * (k.shape[-1] ** -0.5)

        # Prevent access to future positions
        weights = weights.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        # Convert scores into probabilities
        weights = F.softmax(weights, dim=-1)

        weights = self.dropout(weights)

        # Generate weighted values
        v = self.value(x)
        output = weights @ v

        return output


# Test single attention head
head_size = n_embd // n_head

single_head = Head(head_size).to(device)

sample_input = torch.randn(batch_size, block_size, n_embd).to(device)

output = single_head(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 32])


In [ ]:

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        # Create multiple attention heads
        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        # Dropout after concatenation
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Run all attention heads
        outputs = [head(x) for head in self.heads]

        # Concatenate outputs
        out = torch.cat(outputs, dim=-1)

        out = self.dropout(out)

        return out


# Test Multi-Head Attention
multi_head = MultiHeadAttention(
    n_head,
    n_embd // n_head
).to(device)

sample_input = torch.randn(
    batch_size, block_size, n_embd
).to(device)

output = multi_head(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [ ]:

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.network = nn.Sequential(

            # Expand dimensions
            nn.Linear(n_embd, 4 * n_embd),

            # Apply ReLU activation
            nn.ReLU(),

            # Reduce dimensions
            nn.Linear(4 * n_embd, n_embd),

            # Apply dropout
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


# Test Feed Forward Network
feed_forward = FeedForward(n_embd).to(device)

sample_input = torch.randn(
    batch_size, block_size, n_embd
).to(device)

output = feed_forward(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [ ]:

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        # Multi-head attention
        self.sa = MultiHeadAttention(n_head, head_size)

        # Feed forward network
        self.ffwd = FeedForward(n_embd)

        # Layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Attention with residual connection
        x = x + self.sa(self.ln1(x))

        # Feed forward with residual connection
        x = x + self.ffwd(self.ln2(x))

        return x


# Test Transformer Block
transformer_block = Block(n_embd, n_head).to(device)

sample_input = torch.randn(
    batch_size, block_size, n_embd
).to(device)

output = transformer_block(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [ ]:

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embedding
        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        # Position embedding
        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head)
            for _ in range(n_layer)
        ])

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Output layer
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(
                module.weight, mean=0.0, std=0.02
            )

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(
                module.weight, mean=0.0, std=0.02
            )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings
        pos = torch.arange(T, device=device)
        pos_emb = self.position_embedding_table(pos)

        # Combine embeddings
        x = tok_emb + pos_emb

        # Pass through Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Predict next-character logits
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Keep only the latest context
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, loss = self(idx_cond)

            # Consider the last character's prediction
            logits = logits[:, -1, :]

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next character
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append predicted character
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# Create GPT model
model = GPTLanguageModel().to(device)

print("GPT model created successfully!")

# Count parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

print("Total parameters:", total_params)


GPT model created successfully!
Total parameters: 767330


In [ ]:

# Get a sample batch
xb, yb = get_batch("train")

# Pass input through GPT
logits, loss = model(xb, yb)

print("Input shape:", xb.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())


Input shape: torch.Size([32, 128])
Logits shape: torch.Size([4096, 98])
Initial loss: 4.6113762855529785


In [ ]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

print("Optimizer created successfully!")
print("Learning rate:", learning_rate)


Optimizer created successfully!
Learning rate: 0.0003


In [ ]:

@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(
                "train" if split == "train" else "val"
            )

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()

    return results


print("Initial train and validation loss:")
print(estimate_loss())


Initial train and validation loss:
{'train': 4.615135669708252, 'val': 4.615190505981445}


In [ ]:

import time

# Start training timer
start_time = time.time()

model.train()

for iter in range(100):

    # Get a training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Reset gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Print progress
    if (iter + 1) % 20 == 0:
        print(
            f"Iteration {iter + 1}/100 | "
            f"Loss: {loss.item():.4f}"
        )

end_time = time.time()

print("\n100 iterations completed!")
print("Time taken:", round(end_time - start_time, 2), "seconds")


Iteration 20/100 | Loss: 3.5318
Iteration 40/100 | Loss: 3.2381
Iteration 60/100 | Loss: 3.0076
Iteration 80/100 | Loss: 2.7849
Iteration 100/100 | Loss: 2.6790

100 iterations completed!
Time taken: 3.1 seconds


In [ ]:

final_losses = estimate_loss()

print("Current model performance")
print("--------------------------")
print("Training loss:", final_losses["train"])
print("Validation loss:", final_losses["val"])


Current model performance
--------------------------
Training loss: 2.6594369411468506
Validation loss: 2.6528146266937256


In [ ]:

import time

remaining_iters = 2900

start_time = time.time()

model.train()

for iter in range(remaining_iters):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    current_iter = iter + 101

    # Print progress every 100 iterations
    if current_iter % 100 == 0:
        print(
            f"Iteration {current_iter}/3000 | "
            f"Loss: {loss.item():.4f}"
        )

    # Save checkpoint every 500 iterations
    if current_iter % 500 == 0:

        torch.save({
            "iteration": current_iter,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss.item()
        }, "storygpt_checkpoint.pth")

        print("Checkpoint saved at iteration", current_iter)


# Save final checkpoint
torch.save({
    "iteration": 3000,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()
}, "storygpt_checkpoint.pth")

print("\nTraining completed!")
print("Total time:", round(time.time() - start_time, 2), "seconds")



Iteration 200/3000 | Loss: 2.4258
Iteration 300/3000 | Loss: 2.3507
Iteration 400/3000 | Loss: 2.2711
Iteration 500/3000 | Loss: 2.2027
Checkpoint saved at iteration 500
Iteration 600/3000 | Loss: 2.0586
Iteration 700/3000 | Loss: 1.9313
Iteration 800/3000 | Loss: 1.7538
Iteration 900/3000 | Loss: 1.7560
Iteration 1000/3000 | Loss: 1.6530
Checkpoint saved at iteration 1000
Iteration 1100/3000 | Loss: 1.6329
Iteration 1200/3000 | Loss: 1.4943
Iteration 1300/3000 | Loss: 1.5657
Iteration 1400/3000 | Loss: 1.4237
Iteration 1500/3000 | Loss: 1.3893
Checkpoint saved at iteration 1500
Iteration 1600/3000 | Loss: 1.3765
Iteration 1700/3000 | Loss: 1.4096
Iteration 1800/3000 | Loss: 1.3351
Iteration 1900/3000 | Loss: 1.3582
Iteration 2000/3000 | Loss: 1.4020
Checkpoint saved at iteration 2000
Iteration 2100/3000 | Loss: 1.3111
Iteration 2200/3000 | Loss: 1.3103
Iteration 2300/3000 | Loss: 1.3430
Iteration 2400/3000 | Loss: 1.2501
Iteration 2500/3000 | Loss: 1.2873
Checkpoint saved at iteration

In [ ]:

final_losses = estimate_loss()

print("FINAL MODEL EVALUATION")
print("----------------------")
print("Training Loss:", round(final_losses["train"], 4))
print("Validation Loss:", round(final_losses["val"], 4))


FINAL MODEL EVALUATION
----------------------
Training Loss: 1.1331
Validation Loss: 1.1413


In [ ]:

final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "vocab_size": vocab_size,
    "chars": chars,
    "stoi": stoi,
    "itos": itos,
    "config": {
        "batch_size": batch_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "training_iterations": 3000
    },
    "final_train_loss": final_losses["train"],
    "final_validation_loss": final_losses["val"]
}

torch.save(
    final_checkpoint,
    "StoryGPT_Final_Checkpoint.pth"
)

print("Final model saved successfully!")

import os
print("Checkpoint size:",
      round(os.path.getsize("StoryGPT_Final_Checkpoint.pth") / (1024 * 1024), 2),
      "MB")


Final model saved successfully!
Checkpoint size: 9.9 MB


In [ ]:

from google.colab import files

files.download("StoryGPT_Final_Checkpoint.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

model.eval()

prompt = "Once upon a time"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated_tokens = model.generate(
    context,
    max_new_tokens=500
)

generated_story = decode(
    generated_tokens[0].tolist()
)

print("GENERATED STORY")
print("================")
print(generated_story)


GENERATED STORY
Once upon a time, there was a big smoves. It was a curian small named Lily. The lifted tream, and saw a big game wings. They squirrelled and said, "Don't like my too prom, you, just weare his proud. One day, Lily's mommy played with their singgue edgentrage of on the deloud it. He decided take the bush like friends bitted the toy tarty three oursanger. The old and made upside outside to make beone he dreed beautiful people find of the skiprortant?Once upon a time, she was a decideating to could read look, of th


In [ ]:

@torch.no_grad()
def generate_with_temperature(prompt, max_new_tokens, temperature):

    model.eval()

    # Encode the prompt
    context = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep the latest context
        idx_cond = context[:, -block_size:]

        # Get model predictions
        logits, loss = model(idx_cond)

        # Get predictions for the last character
        logits = logits[:, -1, :]

        # Apply temperature
        logits = logits / temperature

        # Convert logits into probabilities
        probabilities = F.softmax(logits, dim=-1)

        # Sample the next character
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Append new character
        context = torch.cat(
            (context, next_token),
            dim=1
        )

    return decode(context[0].tolist())


# Compare different temperatures
prompt = "Once upon a time"

for temp in [0.5, 0.8, 1.0]:

    print("\n" + "=" * 60)
    print("TEMPERATURE:", temp)
    print("=" * 60)

    story = generate_with_temperature(
        prompt,
        max_new_tokens=400,
        temperature=temp
    )

    print(story)



TEMPERATURE: 0.5
Once upon a time, there was a little girl named Lily. She loved to play with his mom and said, "What is friends that working, but it was a big stick and so much he had to be the cooke and made the green friends. They all loved to play with him to make it toys the bear. They were playing with the big friends and said, "This is is sing the bird. It is a big big big had for the bear." 

The animals smiled and said, 

TEMPERATURE: 0.8
Once upon a time, there was a little girl named Ben. She was very happy and his ground her friends. It was that he blother and he wanted to keep him. He was too proud of the find sack, the pige sun green could betty the gring home.Once upon a time, there was a little boy named I tall boy. It little buy was so happy and watch furty. They toped the girl girl of the trees words and down on the little birds.

After s

TEMPERATURE: 1.0
Once upon a time, there was a litre girl named Lily. He loved to play, but the few preture the sky the right roo

In [ ]:

test_prompts = [
    "Once upon a time",
    "One day, a little girl",
    "A little boy went to",
    "One sunny morning"
]

for prompt in test_prompts:

    print("\n" + "=" * 60)
    print("PROMPT:", prompt)
    print("=" * 60)

    story = generate_with_temperature(
        prompt,
        max_new_tokens=300,
        temperature=0.5
    )

    print(story)



PROMPT: Once upon a time
Once upon a time, there was a little girl named Timmy. She loved to play with his friends toys. She liked to play with her friends and they saw the sky. They were saw a big tree friends. They were safe and they had a big smile. They were very and said, "That is that is is the different. I loved to started to the st

PROMPT: One day, a little girl
One day, a little girl named Lily. She was so happy. She loved to play with her friends. He was so happy and dad, so he was so happy to see and said, "I am beautiful and a big big said while green he was so happy the rain and lots of the other. They were started to be a happy could anything pick with his happy.Once upon 

PROMPT: A little boy went to
A little boy went to play. They felt the tree and the story saw the treet. They went to the boy and said, "That we they are toys." They were their happy. They got best their toys and coloud the bite big and small and her friend the water. They saw a little girl named Ben.

In [ ]:

# Final sample generation

final_prompt = "Once upon a time, there was a little girl"

final_story = generate_with_temperature(
    prompt=final_prompt,
    max_new_tokens=700,
    temperature=0.5
)

print("STORYGPT - FINAL GENERATED STORY")
print("=" * 60)
print(final_story)


STORYGPT - FINAL GENERATED STORY
Once upon a time, there was a little girl named Lily. She nodded to play with her friends. She was very friends and they saw a greentle bear. He came on the bowl and said, "What is a started toy with the flower and the bird and looked to cold anymore. It was too have a new grass. It was things on the bird and was very happy. He was so happy to go the bird and said, "Okay, I will have your before." 

The dog and started to play with her friends and the sky. It was a big started to pretend. The copole was little bird of the sky. They were playing with the story. They roome are their beautiful toys. They had to go outside to his friends and toys. They ran to the little soft and couldn't was the party.

The story and the knew the floor
